# Simplified Quant Trading System

This notebook runs the complete workflow: market data processing, indicators, trading signals, execution simulation, and performance evaluation.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

def _find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    return start

ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_data, compute_stats
from src.indicators import add_returns, add_volatility, add_volume_avg
from src.signals import ma_crossover, mean_reversion
from src.execution import simulate_market, simulate_limit, apply_spread
from src.evaluate import (
    buy_and_hold,
    metrics,
    plot_cumulative_returns,
    plot_equity_vs_bh,
    plot_price_signals,
    plot_strategy_comparison,
    plot_volume,
    plot_volatility,
)

STARTING_CASH = 10_000
SPREAD = 0.10
df = load_data(source='csv', path=ROOT / 'data' / 'synthetic_ohlcv.csv')
stats = compute_stats(df)
print(f"Rows: {stats['rows']}")
print(f"Date range: {stats['start_date']} to {stats['end_date']}")
print(f"Mean close: ${stats['close_mean']:,.2f}")
print(f"Average volume: {stats['volume_mean']:,.0f}")


## Market Data Processing

The dataset follows the OHLCV contract and uses business-day timestamps. Missing or invalid values are cleaned before indicators are calculated.


In [ ]:
base = add_volume_avg(add_volatility(add_returns(df), window=20), window=20)
ma_df = ma_crossover(base, short=20, long=50)
ma_trades, ma_equity = simulate_market(ma_df, cash=STARTING_CASH, spread=SPREAD)
ma_bh = buy_and_hold(ma_df, cash=STARTING_CASH)
ma_results = metrics(ma_equity, ma_trades, ma_df)

mr_df = mean_reversion(base, window=20, z=1.35)
mr_trades, mr_equity = simulate_market(mr_df, cash=STARTING_CASH, spread=SPREAD)
mr_results = metrics(mr_equity, mr_trades, mr_df)

print(f"MA final equity: ${ma_results['final_equity']:,.2f}")
print(f"MA return: {ma_results['strategy_return']:.2%}")
print(f"Mean Reversion final equity: ${mr_results['final_equity']:,.2f}")
print(f"Mean Reversion return: {mr_results['strategy_return']:.2%}")
print(f"Buy and Hold final equity: ${ma_bh.iloc[-1]:,.2f}")
print(f"Market trades: {len(ma_trades)}")

plot_volume(base)
plt.show()


## Indicators and Signals

The Moving Average Crossover strategy buys when the short moving average rises above the long moving average and sells when it falls back below. The Mean Reversion strategy buys when price is far below a rolling mean and exits when price is far above it.


In [ ]:
plot_price_signals(ma_df)
plt.show()

plot_volatility(ma_df)
plt.show()


## Execution and Evaluation

Market orders fill at the ask for BUY orders and at the bid for SELL orders. The strategy shifts position by one bar so a signal observed today can only trade on the next available bar.


In [ ]:
plot_equity_vs_bh(ma_equity, ma_bh)
plt.show()

plot_cumulative_returns(ma_equity, ma_bh)
plt.show()

plot_strategy_comparison({
    'MA crossover': ma_equity,
    'Mean reversion': mr_equity,
    'Buy and hold': ma_bh,
})
plt.show()

spread_df = apply_spread(ma_df, SPREAD)
no_fill_limit = spread_df['ask'].min() - 1
no_fill_trades, _ = simulate_limit(spread_df, no_fill_limit, side='buy', cash=STARTING_CASH)
fill_limit = spread_df['ask'].quantile(0.10)
fill_trades, _ = simulate_limit(spread_df, fill_limit, side='buy', cash=STARTING_CASH)

print(f'Limit no-fill trades: {len(no_fill_trades)}')
print(f'Limit fill trades: {len(fill_trades)}')


## Observations

Mean Reversion led this sample, while Moving Average Crossover finished ahead of Buy and Hold because it avoided some weaker periods. The passive baseline stayed fully invested and carried the deepest drawdown. Limit orders illustrate execution uncertainty because an order can remain unfilled when the ask never reaches the chosen limit price.
